# 🧠 EX48: Mean Average Precision (mAP)
### *A Lecture by your Computer Vision Professor*

Welcome back, class! Today, we are going to study the gold-standard metric for object detection: **Mean Average Precision (mAP)**.

In object detection, evaluating a model is significantly more complex than standard classification because we must simultaneously evaluate:
1. **Classification accuracy:** Did we identify the correct class?
2. **Localization precision:** Did we draw the bounding box accurately?

Let's break down the metric from first-principles math all the way to visualization.

---

## 📖 1. The Core Metrics from First Principles

Let's recall the standard definition of precision and recall:

- **Precision (P):** Out of all predictions the model made, what fraction was actually correct?
  $$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
- **Recall (R):** Out of all actual objects in the image, what fraction did the model successfully find?
  $$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{\text{TP}}{\text{Total Ground Truths}}$$

### What constitutes a "True Positive" (TP) in Object Detection?
A predicted bounding box $B_{\text{pred}}$ is a **True Positive (TP)** if and only if:
1. The predicted class label matches the ground-truth class label.
2. The spatial overlap, measured by **Intersection over Union (IoU)**, with a ground-truth box $B_{\text{gt}}$ is greater than or equal to a set threshold $\tau$ (typically $\tau = 0.5$):
   $$\text{IoU}(B_{\text{pred}}, B_{\text{gt}}) = \frac{\text{Area}(B_{\text{pred}} \cap B_{\text{gt}})}{\text{Area}(B_{\text{pred}} \cup B_{\text{gt}}) } \ge \tau$$
3. It is the **highest-confidence prediction** matching that ground-truth box. If multiple predicted boxes match the same ground-truth box, only the one with the highest confidence is labeled a **TP**. All other matching predictions are labeled **False Positives (FPs)**. This is a critical detail to prevent models from spamming boxes to get high recall.

If a prediction fails any of these conditions, it is a **False Positive (FP)**.
A ground-truth box that is not matched by any prediction is a **False Negative (FN)**.

---

## 📐 2. The Area-Under-The-Curve and Interpolation Math

To calculate Average Precision (AP) for a single class:
1. We sort all predictions for that class by confidence score in descending order.
2. We calculate cumulative TP and FP down the sorted list.
3. We compute the Precision and Recall values at each prediction step.
4. We construct a Precision-Recall (PR) curve.

Because the raw PR curve has a "zig-zag" shape, we smooth it using **Interpolated Precision**:
$$P_{\text{interp}}(R) = \max_{\tilde{R} \ge R} P(\tilde{R})$$
This means that for any recall level $R$, we take the maximum precision found for any recall greater than or equal to $R$.

### Method A: 11-Point Interpolation (PASCAL VOC 2007)
We average the interpolated precision at 11 equally spaced recall levels: $0.0, 0.1, 0.2, \dots, 1.0$.
$$\text{AP}_{11} = \frac{1}{11} \sum_{r \in \{0.0, 0.1, \dots, 1.0\}} P_{\text{interp}}(r)$$

### Method B: All-Point Interpolation (COCO & PASCAL VOC 2012)
Instead of sampling 11 points, we integrate over the entire area under the smoothed curve. We find all unique recall points where the precision drops, dividing the area into rectangles:
$$\text{AP} = \sum_{i} (R_{i+1} - R_i) P_{\text{interp}}(R_{i+1})$$

### From AP to mAP
Mean Average Precision (mAP) is simply the average of the APs across all $C$ classes in the dataset:
$$\text{mAP} = \frac{1}{C} \sum_{c=1}^{C} \text{AP}_c$$

---

## 🔄 3. YOLO mAP Variations: mAP@0.5 vs. mAP@0.5:0.95

In modern YOLO logs, you will see two main mAP metrics:
- **mAP@0.5 (mAP50):** The mAP calculated at a single IoU threshold of 0.5. This measures how well the model detects and classifies objects (very lenient on exact boundary alignment).
- **mAP@0.5:0.95 (mAP50-95):** The mAP averaged over 10 different IoU thresholds from 0.50 to 0.95 with steps of 0.05 (i.e., 0.50, 0.55, 0.60, ..., 0.95). This is the standard COCO metric and is much stricter, penalizing models that don't align bounding boxes precisely.

Let's write code to implement this math and visualize the results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def calculate_iou(box1, box2):
    """
    Calculates Intersection over Union (IoU) between box1 and box2.
    Format: [x1, y1, x2, y2]
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    intersection = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union = area1 + area2 - intersection
    if union <= 0.0:
        return 0.0
    return intersection / union

def calculate_ap_single_class(gt_boxes, pred_boxes, iou_threshold=0.5, method="all-point"):
    """
    Calculates AP, cumulative precisions, and cumulative recalls for a single class.
    """
    num_gts = len(gt_boxes)
    num_preds = len(pred_boxes)
    
    if num_gts == 0:
        return 0.0, np.zeros(num_preds), np.zeros(num_preds)
    if num_preds == 0:
        return 0.0, np.array([]), np.array([])
        
    # Sort predictions by confidence score descending
    sorted_preds = sorted(pred_boxes, key=lambda x: x["confidence"], reverse=True)
    
    # Group ground truths by image_id
    gt_by_img = {}
    for gt in gt_boxes:
        img_id = gt["image_id"]
        if img_id not in gt_by_img:
            gt_by_img[img_id] = []
        gt_by_img[img_id].append(gt)
        
    # Track matched ground truths to prevent double-matching
    gt_matched = {}
    for img_id, boxes in gt_by_img.items():
        gt_matched[img_id] = [False] * len(boxes)
        
    tps = np.zeros(num_preds)
    fps = np.zeros(num_preds)
    
    # Determine TP/FP for each prediction
    for idx, pred in enumerate(sorted_preds):
        img_id = pred["image_id"]
        pred_box = pred["box"]
        
        if img_id not in gt_by_img or len(gt_by_img[img_id]) == 0:
            fps[idx] = 1.0
            continue
            
        best_iou = -1.0
        best_gt_idx = -1
        
        for gt_idx, gt in enumerate(gt_by_img[img_id]):
            iou = calculate_iou(pred_box, gt["box"])
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
                
        if best_iou >= iou_threshold:
            if not gt_matched[img_id][best_gt_idx]:
                tps[idx] = 1.0
                gt_matched[img_id][best_gt_idx] = True
            else:
                fps[idx] = 1.0
        else:
            fps[idx] = 1.0
            
    cum_tps = np.cumsum(tps)
    cum_fps = np.cumsum(fps)
    
    precisions = cum_tps / (cum_tps + cum_fps)
    recalls = cum_tps / num_gts
    
    ap = 0.0
    if method == "11-point":
        for t in np.linspace(0.0, 1.0, 11):
            matching_indices = np.where(recalls >= t)[0]
            p_at_t = np.max(precisions[matching_indices]) if len(matching_indices) > 0 else 0.0
            ap += p_at_t / 11.0
    else:
        mrec = np.concatenate(([0.0], recalls, [1.0]))
        mpre = np.concatenate(([0.0], precisions, [0.0]))
        
        for i in range(len(mpre) - 2, -1, -1):
            mpre[i] = max(mpre[i], mpre[i+1])
            
        i = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[i+1] - mrec[i]) * mpre[i+1])
        
    return ap, precisions, recalls

## 📊 4. PR-Curve & Box Match Visualizations

Let's evaluate the single-class verification dataset, compute its AP, and plot:
1. **The Precision-Recall Curve (Raw vs. Interpolated)**.
2. **Bounding Box Matches** to visually see how predictions map to ground truth.

In [ ]:
# Verification data
gt_boxes = [
    {"image_id": 0, "box": [10, 10, 50, 50]},
    {"image_id": 0, "box": [60, 60, 100, 100]},
    {"image_id": 0, "box": [120, 120, 160, 160]}
]

pred_boxes = [
    {"image_id": 0, "box": [12, 12, 48, 48], "confidence": 0.95}, # TP
    {"image_id": 0, "box": [58, 58, 98, 98], "confidence": 0.88}, # TP
    {"image_id": 0, "box": [122, 122, 158, 158], "confidence": 0.75}, # TP
    {"image_id": 0, "box": [14, 14, 46, 46], "confidence": 0.65}, # FP (Duplicate)
    {"image_id": 0, "box": [200, 200, 240, 240], "confidence": 0.50}  # FP (No overlap)
]

ap, prec, rec = calculate_ap_single_class(gt_boxes, pred_boxes, iou_threshold=0.5, method="all-point")

# Create the dual-plot visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Precision-Recall Curve
# Interpolated curve coordinates for step plot
mrec = np.concatenate(([0.0], rec, [1.0]))
mpre = np.concatenate(([0.0], prec, [0.0]))
for i in range(len(mpre) - 2, -1, -1):
    mpre[i] = max(mpre[i], mpre[i+1])

axes[0].step(mrec, mpre, where='post', label='Interpolated PR Curve', color='#2ca02c', linewidth=3)
axes[0].plot(rec, prec, 'o--', label='Raw Predictions', color='#1f77b4', markersize=8, alpha=0.7)
axes[0].fill_between(mrec, mpre, step='post', alpha=0.15, color='#2ca02c')

axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title(f'Precision-Recall Curve (AP = {ap:.4f})', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 1.05)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(fontsize=11)

# Plot 2: Bounding Box Matches
axes[1].set_xlim(0, 250)
axes[1].set_ylim(250, 0) # Invert Y axis to mimic image coordinates
axes[1].set_title('Spatial Visualization of Bounding Boxes', fontsize=14, fontweight='bold')

# Draw Ground Truths (Green)
for idx, gt in enumerate(gt_boxes):
    box = gt["box"]
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                            linewidth=2, edgecolor='#2ca02c', facecolor='none', linestyle='-')
    axes[1].add_patch(rect)
    axes[1].text(box[0]+2, box[1]+15, f"GT {idx+1}", color='#2ca02c', fontweight='bold', fontsize=10)

# Draw Predictions (Blue/Red)
for idx, pred in enumerate(pred_boxes):
    box = pred["box"]
    conf = pred["confidence"]
    is_tp = idx in [0, 1, 2]
    edge_color = '#1f77b4' if is_tp else '#d62728'
    label = f"Pred {idx+1} ({conf:.2f}) [TP]" if is_tp else f"Pred {idx+1} ({conf:.2f}) [FP]"
    
    rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                            linewidth=2, edgecolor=edge_color, facecolor='none', linestyle='--')
    axes[1].add_patch(rect)
    axes[1].text(box[0]+2, box[3]-5, label, color=edge_color, fontweight='bold', fontsize=9)

axes[1].grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()